### SET UP DATABASE

In [4]:
import mysql.connector
import pandas as pd

def setup_database():
    # Database configuration
    config = {
        'host': 'Hashir',
        'user': 'root',
        'password': 'datascience'
    }
    
    try:
        # Connect to MySQL
        conn = mysql.connector.connect(**config)
        cursor = conn.cursor()
        
        # Create database
        cursor.execute("create database if not exists ABCD_DW")
        cursor.execute("use ABCD_DW")
        print("Database created successfully")
        
        # Read and execute SQL script
        with open('Create-DW.sql', 'r') as file:
            sql_script = file.read()
        
        # Execute each statement
        statements = sql_script.split(';')
        for statement in statements:
            if statement.strip():
                cursor.execute(statement)
        
        print("Schema created successfully")
        
        # Load master data
        load_master_data(conn)
        
        conn.commit()
        conn.close()
        print("Database setup completed successfully")
        
    except Exception as e:
        print(f"Error setting up database: {e}")

def load_master_data(conn):
    # Loading customer and product master data into dimension tables
    try:
        cursor = conn.cursor()
        
        # Load customer master data
        print("Loading customer master data")
        customer_df = pd.read_csv('customer_master_data.csv')
        for _, row in customer_df.iterrows():
            query = """
            insert into DimCustomer 
            (customer_id, gender, age_group, occupation, city_category, stay_in_current_city_years, marital_status)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
            """
            values = (
                row['Customer_ID'], row['Gender'], row['Age'], 
                row['Occupation'], row['City_Category'], 
                row['Stay_In_Current_City_Years'], row['Marital_Status']
            )
            cursor.execute(query, values)
        
        # Loading product master data and create store records
        print("Loading product and store data.")
        product_df = pd.read_csv('product_master_data.csv')
        
        # creating unique store entries
        stores = product_df[['storeID', 'storeName', 'supplierID', 'supplierName']].drop_duplicates()
        store_keys = {}
        
        for _, store_row in stores.iterrows():
            store_query = """
            insert into DimStore (store_id, store_name, supplier_id, supplier_name)
            VALUES (%s, %s, %s, %s)
            """
            store_values = (
                store_row['storeID'], store_row['storeName'], 
                store_row['supplierID'], store_row['supplierName']
            )
            cursor.execute(store_query, store_values)
            store_keys[store_row['storeID']] = cursor.lastrowid
        
        # Then load products
        for _, product_row in product_df.iterrows():
            product_query = """
            insert into DimProduct (product_id, product_category, price)
            VALUES (%s, %s, %s)
            """
            product_values = (
                product_row['Product_ID'], product_row['Product_Category'], 
                product_row['price$']
            )
            cursor.execute(product_query, product_values)
        
        print("Master data loaded successfully")
        
    except Exception as e:
        print(f"Error loading master data: {e}")

if __name__ == "__main__":
    setup_database()

Database created successfully
Schema created successfully
Loading customer master data
Loading product and store data.
Master data loaded successfully
Database setup completed successfully


### HYBRID JOIN 

In [7]:
import pandas as pd
import mysql.connector
from datetime import datetime

class Batchhybridjoin:
    def __init__(self, db_config, batch_size=1000):
        self.db_config = db_config
        self.batch_size = batch_size
        self.processed_count = 0
        
        # Database connection
        try:
            self.conn = mysql.connector.connect(**db_config)
            self.cursor = self.conn.cursor()
            print("Database connection established successfully")
        except Exception as e:
            print(f"Database connection failed: {e}")
            raise
        
        # Load all master data 
        self.load_master_data()
    
    def load_master_data(self):
        # Loading all master data into memory
        print("Loading master data into memory")
        
        try:
            # Load customer master data (disk relation R1)
            self.customer_df = pd.read_csv('customer_master_data.csv')
            self.customer_dict = self.customer_df.set_index('Customer_ID').to_dict('index')
            
            # Load product master data (disk relation R2)  
            self.product_df = pd.read_csv('product_master_data.csv')
            self.product_dict = self.product_df.set_index('Product_ID').to_dict('index')
            
            print(f"Loaded {len(self.customer_dict)} customers and {len(self.product_dict)} products")
        except Exception as e:
            print(f"Error loading master data: {e}")
            raise
    
    def hybrid_join_enrichment(self, transaction):
        # hybridjoin operation: Join transactional data with both customer and product master data
        # This simulates the stream relation join operation described in the project
        
        try:
            customer_id = transaction['customer_id']
            product_id = transaction['product_id']
            
            # hybridjoin Step: Probe disk relations (customer and product master data)
            customer_data = self.customer_dict.get(customer_id)
            product_data = self.product_dict.get(product_id)
            
            if not customer_data or not product_data:
                return None
            
            # Get dimension keys (this represents the JOIN results)
            customer_key = self.get_or_create_customer_key(customer_id)
            product_key = self.get_or_create_product_key(product_id)
            time_key = self.get_or_create_time_key(transaction['date'])
            store_key = self.get_or_create_store_key(product_data)
            
            if None in [customer_key, product_key, time_key, store_key]:
                return None
            
            # Calculate total amount using enriched data
            price = product_data.get('price$', 0)
            total_amount = transaction['quantity'] * price
            
            # Return fully enriched record (transaction + customer + product data)
            enriched = {
                'customer_key': customer_key,
                'product_key': product_key,
                'store_key': store_key,
                'time_key': time_key,
                'order_id': transaction['order_id'],
                'quantity': transaction['quantity'],
                'total_amount': round(total_amount, 2),
                # Additional enriched fields from hybridjoin
                'gender': customer_data.get('Gender'),
                'age_group': customer_data.get('Age'),
                'occupation': customer_data.get('Occupation'),
                'city_category': customer_data.get('City_Category'),
                'product_category': product_data.get('Product_Category'),
                'price': price,
                'store_id': product_data.get('storeID'),
                'supplier_id': product_data.get('supplierID')
            }
            
            return enriched
            
        except Exception as e:
            print(f"Error in hybridjoin enrichment: {e}")
            return None
    
    def get_or_create_time_key(self, date_str):
        # Get or create time dimension record
        try:
            date_obj = datetime.strptime(date_str, '%Y-%m-%d')
            year = date_obj.year
            month = date_obj.month
            day = date_obj.day
            quarter = (month - 1) // 3 + 1
            day_of_week = date_obj.weekday()
            is_weekend = 1 if day_of_week >= 5 else 0
            
            query = "select time_key from DimTime where full_date = %s"
            self.cursor.execute(query, (date_str,))
            result = self.cursor.fetchone()
            
            if result:
                return result[0]
            else:
                insert_query = """
                insert into DimTime (full_date, day, month, year, quarter, day_of_week, is_weekend)
                VALUES (%s, %s, %s, %s, %s, %s, %s)
                """
                values = (date_str, day, month, year, quarter, day_of_week, is_weekend)
                self.cursor.execute(insert_query, values)
                self.conn.commit()
                return self.cursor.lastrowid
                
        except Exception as e:
            print(f"Error in get_or_create_time_key: {e}")
            return None
    
    def get_or_create_customer_key(self, customer_id):
        # Get or create customer dimension record
        try:
            query = "select customer_key from DimCustomer where customer_id = %s"
            self.cursor.execute(query, (customer_id,))
            result = self.cursor.fetchone()
            
            if result:
                return result[0]
            else:
                customer_data = self.customer_dict.get(customer_id)
                if customer_data:
                    insert_query = """
                    insert into DimCustomer (customer_id, gender, age_group, occupation, city_category, stay_in_current_city_years, marital_status)
                    VALUES (%s, %s, %s, %s, %s, %s, %s)
                    """
                    values = (
                        customer_id,
                        customer_data['Gender'],
                        customer_data['Age'],
                        customer_data['Occupation'],
                        customer_data['City_Category'],
                        customer_data['Stay_In_Current_City_Years'],
                        customer_data['Marital_Status']
                    )
                    self.cursor.execute(insert_query, values)
                    self.conn.commit()
                    return self.cursor.lastrowid
                return None
        except Exception as e:
            print(f"Error in get_or_create_customer_key: {e}")
            return None
    
    def get_or_create_product_key(self, product_id):
        # Get or create product dimension record
        try:
            query = "select product_key from DimProduct where product_id = %s"
            self.cursor.execute(query, (product_id,))
            result = self.cursor.fetchone()
            
            if result:
                return result[0]
            else:
                product_data = self.product_dict.get(product_id)
                if product_data:
                    insert_query = """
                    insert into DimProduct (product_id, product_category, price)
                    VALUES (%s, %s, %s)
                    """
                    values = (
                        product_id,
                        product_data['Product_Category'],
                        product_data['price$']
                    )
                    self.cursor.execute(insert_query, values)
                    self.conn.commit()
                    return self.cursor.lastrowid
                return None
        except Exception as e:
            print(f"Error in get_or_create_product_key: {e}")
            return None
    
    def get_or_create_store_key(self, product_data):
        # Get or create store dimension record
        try:
            store_id = product_data.get('storeID')
            supplier_id = product_data.get('supplierID')
            
            if store_id is None or supplier_id is None:
                return None
            
            query = "select store_key from DimStore where store_id = %s AND supplier_id = %s"
            self.cursor.execute(query, (store_id, supplier_id))
            result = self.cursor.fetchone()
            
            if result:
                return result[0]
            else:
                insert_query = """
                insert into DimStore (store_id, store_name, supplier_id, supplier_name)
                VALUES (%s, %s, %s, %s)
                """
                values = (
                    store_id,
                    product_data.get('storeName'),
                    supplier_id,
                    product_data.get('supplierName')
                )
                self.cursor.execute(insert_query, values)
                self.conn.commit()
                return self.cursor.lastrowid
        except Exception as e:
            print(f"Error in get_or_create_store_key: {e}")
            return None
    
    def load_to_dw(self, enriched_record):
        # Load enriched record to fact table
        try:
            query = """
            insert into FactSales 
            (customer_key, product_key, store_key, time_key, order_id, quantity, total_amount)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
            """
            values = (
                enriched_record['customer_key'],
                enriched_record['product_key'],
                enriched_record['store_key'],
                enriched_record['time_key'],
                enriched_record['order_id'],
                enriched_record['quantity'],
                enriched_record['total_amount']
            )
            
            self.cursor.execute(query, values)
            self.conn.commit()
            return True
            
        except Exception as e:
            print(f"Error loading to DW: {e}")
            self.conn.rollback()
            return False
    
    def process_batch_hybrid_join(self, transactional_file):
        
        # Main hybridjoin batch processing algorithm
        # Implements the hybridjoin concept by joining stream data with multiple disk relations
        
        print(f"Processing {transactional_file} using Batch hybridjoin")
        print("hybridjoin: Joining transactional stream with customer and product master data")
        
        try:
            # Extract phase: Read transactional data (stream S)
            transactions_df = pd.read_csv(transactional_file)
            total_records = len(transactions_df)
            processed_count = 0
            failed_count = 0
            
            print(f"Total records to process: {total_records}")
            
            # Process in batches (simulating stream processing with disk relation probing)
            for start_idx in range(0, total_records, self.batch_size):
                end_idx = min(start_idx + self.batch_size, total_records)
                batch_df = transactions_df.iloc[start_idx:end_idx]
                
                batch_processed = 0
                batch_failed = 0
                
                # hybridjoin Transformation: Enrich each transaction with master data
                for _, row in batch_df.iterrows():
                    try:
                        # Extract transaction from stream
                        transaction = {
                            'order_id': int(row['orderID']),
                            'customer_id': int(row['Customer_ID']),
                            'product_id': row['Product_ID'],
                            'quantity': int(row['quantity']),
                            'date': row['date']
                        }
                        
                        
                        # hybridjoin Operation: Transform by joining with master data
                        # This simulates probing disk relations R1 (customer) and R2 (product)
                        enriched = self.hybrid_join_enrichment(transaction)
                        
                        print(enriched)
                        # Load to data warehouse
                        if enriched and self.load_to_dw(enriched):
                            batch_processed += 1
                        else:
                            batch_failed += 1
                            
                    except Exception as e:
                        print(f"Error processing record {row['orderID']}: {e}")
                        batch_failed += 1
                        continue
                
                processed_count += batch_processed
                failed_count += batch_failed
                
                print(f"Batch {start_idx//self.batch_size + 1}: "
                      f"hybridjoin processed {batch_processed} records, "
                      f"Failed {batch_failed} records "
                      f"(Total: {processed_count}/{total_records})")
            
            print(f"hybridjoin processing completed")
            print(f"Successfully enriched and loaded: {processed_count} records")
            print(f"Failed records: {failed_count}")
            return processed_count
            
        except Exception as e:
            print(f"Error in hybridjoin processing: {e}")
            return 0
    
    def close(self):
        # Close database connection
        if self.conn:
            self.conn.close()
            print("Database connection closed.")

# Main execution
if __name__ == "__main__":
    host = "Hashir"
    user = "root"
    password = "datascience"
    database = "ABCD_DW"
    
    db_config = {
        'host': host,
        'user': user,
        'password': password,
        'database': database
    }
    
    try:
        # Initialize and run Batch hybridjoin
        processor = Batchhybridjoin(db_config, batch_size=1000)
        processed_count = processor.process_batch_hybrid_join('transactional_data.csv')
        processor.close()
        
        print(f"hybridjoin ETL process completed successfully")
        print(f"Total records joined with master data and loaded into DW: {processed_count}")
        
    except Exception as e:
        print(f"Error: {e}")
        print("Please check your database credentials and ensure the database exists.")

Database connection established successfully
Loading master data into memory
Loaded 5891 customers and 3631 products
Processing transactional_data.csv using Batch hybridjoin
hybridjoin: Joining transactional stream with customer and product master data
Total records to process: 550068
{'customer_key': 1, 'product_key': 1, 'store_key': 1, 'time_key': 1, 'order_id': 9914432, 'quantity': 1, 'total_amount': 77.51, 'gender': 'F', 'age_group': '0-17', 'occupation': 10, 'city_category': 'A', 'product_category': 'Home & Kitchen', 'price': 77.51, 'store_id': 2, 'supplier_id': 13}
{'customer_key': 1, 'product_key': 2, 'store_key': 2, 'time_key': 2, 'order_id': 1676537, 'quantity': 3, 'total_amount': 28.89, 'gender': 'F', 'age_group': '0-17', 'occupation': 10, 'city_category': 'A', 'product_category': 'Grocery', 'price': 9.63, 'store_id': 6, 'supplier_id': 9}
{'customer_key': 1, 'product_key': 3, 'store_key': 3, 'time_key': 3, 'order_id': 8910457, 'quantity': 1, 'total_amount': 31.66, 'gender': '

KeyboardInterrupt: 